In [1]:
import numpy as np
from netCDF4 import Dataset
import matplotlib.pyplot as plt
#import cartopy.crs as crs
from cartopy.io.shapereader import Reader
from cartopy.feature import ShapelyFeature
from cartopy.feature import NaturalEarthFeature
import cartopy.crs as ccrs
from datetime import datetime, timedelta
from metpy.units import units
from metpy.calc import dewpoint_from_specific_humidity, relative_humidity_from_specific_humidity, wind_speed, specific_humidity_from_mixing_ratio
import pyart
from wrf import getvar
import haversine
from metpy.interpolate import log_interpolate_1d, interpolate_1d


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



In [2]:
wspd_thresh = 20 #wind speed cutoff for UAS ops, in m/s

#dt = datetime(2022,9,15,10)
#dt = datetime(2022,7,19,10)
dt = datetime(2021,6,4,10)

#ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100mIOP4.nc')
ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100mnew.nc')

location = ncfile1.variables['location'][:]
qc = ncfile1.variables['qc'][:]
obstype = ncfile1.variables['obs_type'][:]
obstypemd = ncfile1.variables['ObsTypesMetaData'][:]
obs_val = ncfile1.variables['observations'][:]
which_vert = ncfile1.variables['which_vert'][:]

print(obstype)
qc_new = []
for i in range(len(qc)):
    qc_d = qc[i][0]
    qc_new.append(qc_d)
qc_new = np.asarray(qc_new)

otype1 = 142
loc_T = location[obstype==otype1, :]
qc_T = qc_new[obstype==otype1]
obs_T = obs_val[obstype==otype1, :]
lons_T = loc_T[:,0]
lats_T = loc_T[:,1]

lons_T[lons_T > 180] = lons_T[lons_T > 180] - 360

prof_lons = lons_T[np.unique(lons_T, return_index=True)[1]]
prof_lats = lats_T[np.unique(lons_T, return_index=True)[1]]
prof_esfc = loc_T[:,2][np.unique(lons_T, return_index=True)[1]]

[142 105 106 ...  26  27  28]


In [3]:
print(prof_lats)
print(prof_lons)

[39.34999847 39.25       39.04000092 39.36000061 39.09999847 39.52999878
 39.45999908 39.59999847 39.08000183]
[-85.26000214 -84.77999878 -84.66999817 -84.51999664 -84.41999817
 -84.40000153 -84.25       -84.23000336 -84.22000122]


In [ ]:
#Make arrays of heights and times
#For 1 km profile at a 3 m/s ascent/descent rate

#profile depth, in m
depth = 1000

#profile resolution, in m
res = 50

#uas_z = np.concatenate([np.arange(res, depth+res, res), np.arange(depth-res, 0, res*-1)])
uas_z = np.concatenate([np.arange(res-res, depth+res, res)])

#Get time, in seconds, for this profile

#ascent rate (in m/s)
ar = 3

#uas_time = np.arange(res/ar, ((len(uas_z)*res)/ar)+(res/ar), res/ar) 
uas_time = np.arange(0, ((len(uas_z)*res)/ar), res/ar) 
print(uas_z)
print(uas_time)

print(uas_z.shape)
print(uas_time.shape)

#Replicate this profile hourly
#Interval between flights (in seconds)
fl_int = 3600
#Number of flights
fl_num = 10
#Start day
#st_day = 154024
#st_day = 153966
st_day = 153556

#first_time (in seconds)
st_sec = 3*fl_int
end_sec = (3+fl_num)*fl_int
#Get a range of times

time_range = np.arange((st_sec/86400), (end_sec/86400), fl_int/86400) + st_day

#Now get the times assuming launches at each interval

uas_times2=[]
uas_z2 = []
for time_l in time_range:
    time_prof = time_l + (uas_time/86400)
    uas_times2.append(time_prof)
    uas_z2.append(uas_z)
uas_times2 = np.concatenate(uas_times2)
uas_z2=np.concatenate(uas_z2)
print(uas_times2)

In [8]:
#Obs types for the UAS obs. Change once we have this implemented in DART
otype_T = 122
otype_q = 123
otype_u = 120
otype_v = 121

#Errors for UAS obs
oerr_T = 1.0
oerr_q = 0.01
oerr_u = 1.0
oerr_v = 1.0

otype_s = []
obs_s = []
lon_s = []
lat_s = []
elev_s = []
error_s = []
time_s = []

minute_range = np.arange(180,725,5)
#minute_range = np.arange(595,605,5)

for mins in minute_range:
    #dt_start = datetime(2022,9,15,0,0)
    #dt_start = datetime(2022,7,19,0,0)
    dt_start = datetime(2021,6,4,0,0)
    
    dt = dt_start + timedelta(minutes=int(mins))
    print(dt)
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP6/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP4/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100JUNE/final_nature/wrfout_d02_2021-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    
    lon = wrfout.variables['XLONG']
    lat = wrfout.variables['XLAT']
    U10 = wrfout.variables['U10']
    V10 = wrfout.variables['U10']
    T2 = np.asarray(wrfout.variables['T2'])*units('K')
    T2F = T2 .to('degF')
    Q2 = np.asarray(wrfout.variables['Q2'])
    P2 = np.asarray(wrfout.variables['PSFC'][:]/100) * units('hPa')
    Td2 = dewpoint_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    RH2 = relative_humidity_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    SPD10 = wind_speed(np.asarray(U10)*units('m/s'), np.asarray(V10)*units('m/s'))
    cloud=wrfout.variables['QCLOUD']
    T_z = np.asarray(getvar(wrfout, "tk"))
    p_z = np.asarray(getvar(wrfout, "pres"))
    q_zi = np.asarray(wrfout.variables['QVAPOR'][:])
    q_z = specific_humidity_from_mixing_ratio(q_zi)
    u_z, v_z = getvar(wrfout, 'uvmet')
    u_z = np.asarray(u_z)
    v_z = np.asarray(v_z)
    z_z = np.asarray(getvar(wrfout, "height_agl"))

    #Convert WRF file time into same units as the obs_seq time
    dt_tot = (dt - datetime(1601,1,1)).total_seconds() / 86400
    time_diff = np.abs(dt_tot - uas_times2)
    
    #Get obs within +- 2.5 minutes of each WRF file
    time_T3 = uas_times2[time_diff<(150/86400)]
    #loc_T3 = loc_T2[time_diff<(150/86400), :]
    #qc_T3 = qc_T2[time_diff<(150/86400)]
    elev_T3 = uas_z2[time_diff<(150/86400)]
    if len(time_T3)==0:
            print('NO OBS IN WINDOW')
            continue
    #Get the correct lat/lon for an example point
    for st1 in range(len(prof_lats)):
        latp=prof_lats[st1]
        lonp=prof_lons[st1]
        esfcp = prof_esfc[st1]
        #Get location for each ob in model land
        lon1d = np.ndarray.flatten(lon[0,:,:])
        lat1d = np.ndarray.flatten(lat[0,:,:])
        station = []
        points = []
        for i in range(len(lon1d)):
            points.append((lat1d[i],lon1d[i]))
            station.append((latp,lonp))
        dist = haversine.haversine_vector(station,points)
        dist2=dist.reshape(lon.shape[1],lon.shape[2])
        print(lon[0,:,:][np.where(dist2==np.min(dist2))])
        print(lat[0,:,:][np.where(dist2==np.min(dist2))])
        print(np.where(dist2==np.min(dist2)))
        st_xind = np.where(dist2==np.min(dist2))[0][0]
        st_yind = np.where(dist2==np.min(dist2))[1][0]

        #Actually get those interpolated obs
        # z_point = z_z[:,st_xind,st_yind]
        # t_point = T_z[:,st_xind,st_yind]
        # q_point = q_z[0,:,st_xind,st_yind]
        # u_point = u_z[:,st_xind,st_yind]
        # v_point = v_z[:,st_xind,st_yind]
        z_point = np.concatenate([[0], z_z[:,st_xind,st_yind]])
        t_point = np.concatenate([[T2[0,st_xind,st_yind].magnitude], T_z[:,st_xind,st_yind]])
        q_point = np.concatenate([[Q2[0,st_xind,st_yind]], q_z[0,:,st_xind,st_yind]])
        u_point = np.concatenate([[U10[0,st_xind,st_yind]], u_z[:,st_xind,st_yind]])
        v_point = np.concatenate([[V10[0,st_xind,st_yind]], v_z[:,st_xind,st_yind]])
        
        m = 0
        for point in elev_T3:
            
            T2_a = interpolate_1d(point, z_point, t_point)
            q2_a = interpolate_1d(point, z_point, q_point)
            u2_a = interpolate_1d(point, z_point, u_point)
            v2_a = interpolate_1d(point, z_point, v_point)

            wspd_a = np.sqrt(u2_a**2 + v2_a**2)
            if wspd_a > wspd_thresh:
                print('wind is too strong, skipping')
            else:
            
                error = np.random.normal(loc=0.0, scale=np.sqrt(oerr_T))
                if np.abs(error/4) > (np.sqrt(oerr_T)*1.0):
                    error = (error / np.abs(error)) * (np.sqrt(oerr_T)*1.0)
                T2_b = T2_a + error/4
                #uas_T.append(T2_b)
                print(T2_b)
                obs_s.append(T2_b)
                otype_s.append(otype_T)
                time_s.append(time_T3[m])
                elev_s.append(point+esfcp)
                error_s.append(oerr_T)
                lon_s.append(lonp)
                lat_s.append(latp)
                
                #Q section
                error = np.random.normal(loc=0.0, scale=np.sqrt(oerr_q))
                if np.abs(error/4) > (np.sqrt(oerr_q)*1.0):
                    error = (error / np.abs(error)) * (np.sqrt(oerr_q)*1.0)
                q2_b = q2_a + error/4
                #uas_q.append(q2_b)
                obs_s.append(q2_b.magnitude)
                otype_s.append(otype_q)
                time_s.append(time_T3[m])
                elev_s.append(point+esfcp)
                error_s.append(oerr_q)
                lon_s.append(lonp)
                lat_s.append(latp)
                
                #U section
                error = np.random.normal(loc=0.0, scale=np.sqrt(oerr_u))
                if np.abs(error/4) > (np.sqrt(oerr_u)*1.0):
                    error = (error / np.abs(error)) * (np.sqrt(oerr_u)*1.0)
                u2_b = u2_a + error/4
                #uas_u.append(u2_b)
                obs_s.append(u2_b)
                otype_s.append(otype_u)
                time_s.append(time_T3[m])
                elev_s.append(point+esfcp)
                error_s.append(oerr_u)
                lon_s.append(lonp)
                lat_s.append(latp)
                
                #V section
                error = np.random.normal(loc=0.0, scale=np.sqrt(oerr_v))
                if np.abs(error/4) > (np.sqrt(oerr_v)*1.0):
                    error = (error / np.abs(error)) * (np.sqrt(oerr_v)*1.0)
                v2_b = v2_a + error/4
                #uas_v.append(v2_b)
                obs_s.append(v2_b)
                otype_s.append(otype_v)
                time_s.append(time_T3[m])
                elev_s.append(point+esfcp)
                error_s.append(oerr_v)
                lon_s.append(lonp)
                lat_s.append(latp)
            
            m=m+1
    

2021-06-04 03:00:00
[-85.260086]
[39.350243]
(array([679]), array([133]))
[291.33290764]
[291.08344419]
[291.18175998]
[290.31215793]
[290.45814513]
[291.07626048]
[291.62073795]
[291.7984441]
[291.90665212]
[-84.77993]
[39.24957]
(array([579]), array([505]))
[289.04933087]
[289.80904652]
[289.09419258]
[289.37100868]
[289.99884878]
[290.37447378]
[290.81929656]
[290.99993374]
[291.022874]


KeyboardInterrupt: 

In [5]:
#Convert lats and lons to radians for DART, because why not
lon_DART = np.radians(np.asarray(lon_s))
lat_DART = np.radians(np.asarray(lat_s))

lon_DART = np.where(lon_DART > 0.0, lon_DART, lon_DART+(2.0*np.pi))

#Convert time into DART format. This is hacky now, improve later
#day_DART = 154024
#day_DART = 153966
day_DART = 153556

seconds_DART = (np.asarray(time_s) - day_DART) * 86400

In [6]:
#Sort everything in time order
inds_time = seconds_DART.argsort()
# print(seconds_DART)
# print(seconds_DART[inds_time])
seconds_DART1 = seconds_DART[inds_time]
seconds_DART1[seconds_DART1 < 0] = 0
obs_s1 = np.asarray(obs_s)[inds_time]
lon_DART1 = lon_DART[inds_time]
lat_DART1 = lat_DART[inds_time]
elev_s1 = np.asarray(elev_s)[inds_time]
otype_s1 = np.asarray(otype_s)[inds_time]
error_s1 = np.asarray(error_s)[inds_time]

In [7]:
for bigfoot in [1,2]:
    print(bigfoot)
    #Write the simulated obs out to an obs_seq file
    filename = 'SIM_UAS_JUNE_prof_up50m'
    fi = open(filename, "w")
    fi.write(" obs_sequence\n")
    fi.write("obs_kind_definitions\n")
    fi.write("    %d \n" %(4))
    fi.write("    %d          %s   \n" %(122, 'UAS_TEMPERATURE'))
    fi.write("    %d          %s   \n" %(123, 'UAS_SPECIFIC_HUMIDITY'))
    fi.write("    %d          %s   \n" %(120, 'UAS_U_WIND_COMPONENT'))
    fi.write("    %d          %s   \n" %(121, 'UAS_V_WIND_COMPONENT'))
    
    fi.write("  num_copies:            %d  num_qc:            %d\n" % (1,1))
    fi.write(" num_obs:       %d  max_num_obs:       %d\n" % (len(obs_s1), len(obs_s1)))
    fi.write("MADIS observation\n")
    fi.write("Data QC\n")
    
    fi.write("  first:            %d  last:       %d\n" % (1, len(obs_s1)))
    
    for q in range(len(obs_s)):
    
        fi.write(" OBS            %d\n" % (q+1) )
        fi.write("   %20.14f\n" % obs_s1[q] )
        fi.write("   %20.14f\n" % 1.0 )
    
        if q+1 == 1:
            fi.write(" %d %d %d\n" % (-1, q+2, -1) ) #First obs
        elif q+1 == len(obs_s):
            fi.write(" %d %d %d\n" % (q, -1, -1) ) #Last obs
        else:
            fi.write(" %d %d %d\n" % (q, q+2, -1) )
    
        fi.write("obdef\n")
        fi.write("loc3d\n")
        fi.write("    %20.14f          %20.14f          %20.14f     %d\n" %
                           (lon_DART1[q], lat_DART1[q], elev_s1[q], 3))
        fi.write("kind\n")
    
        fi.write("     %d     \n" % otype_s1[q])
    
        fi.write("    %d          %d     \n" % (seconds_DART1[q], day_DART))
    
        fi.write("    %20.14f  \n" % error_s1[q])

1
2


/glade/derecho/scratch/mawilson/tmp/ipykernel_4461/151819536.py:24: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  fi.write("   %20.14f\n" % obs_s1[q] )


In [8]:
print((st_sec/86400))

0.125


In [9]:
print(fl_num*(st_sec/86400))

1.25
